In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ayush3954/fundus-images/Original Dataset/Original Dataset/Macular Scar/Macular Scar333.jpg
/kaggle/input/datasets/ayush3954/fundus-images/Original Dataset/Original Dataset/Macular Scar/Macular Scar119.jpg
/kaggle/input/datasets/ayush3954/fundus-images/Original Dataset/Original Dataset/Macular Scar/Macular Scar417.jpg
/kaggle/input/datasets/ayush3954/fundus-images/Original Dataset/Original Dataset/Macular Scar/Macular Scar73.jpg
/kaggle/input/datasets/ayush3954/fundus-images/Original Dataset/Original Dataset/Macular Scar/Macular Scar308.jpg
/kaggle/input/datasets/ayush3954/fundus-images/Original Dataset/Original Dataset/Macular Scar/Macular Scar358.jpg
/kaggle/input/datasets/ayush3954/fundus-images/Original Dataset/Original Dataset/Macular Scar/Macular Scar413.jpg
/kaggle/input/datasets/ayush3954/fundus-images/Original Dataset/Original Dataset/Macular Scar/Macular Scar419.jpg
/kaggle/input/datasets/ayush3954/fundus-images/Original Dataset/Original Dataset/Macular 

In [2]:
   import os
   print(os.listdir('/kaggle/input'))

['datasets']


In [3]:
"""
analyze_dataset.py  (Kaggle Notebook version)

Day 1 script: analyze the 10-class fundus dataset before doing anything else.

KAGGLE NOTES:
  - Input datasets live under /kaggle/input/<dataset-name>/ and are READ-ONLY.
  - The only writable directory in a Kaggle notebook is /kaggle/working/
    (session disk quota is limited and typically wiped on restart unless you
    explicitly save it as output).
  - Because /kaggle/input/ is read-only and your dataset is ~1.5GB, this
    script SYMLINKS files into the train/val/test split by default instead
    of copying them — copying would double your disk usage for no benefit.
    Pass --copy if you specifically want real copies instead.
  - Defaults below assume you added the dataset via "Add Input" in the
    notebook UI. Adjust DEFAULT_DATA_DIR if your dataset folder name differs
    (check the exact path by running: ls /kaggle/input/ in a cell first).

What this does:
  1. Auto-detects your dataset's folder structure (folder-per-class is assumed
     first, since that's the most common layout for downloaded classification
     datasets — falls back to CSV/JSON label file if no class folders found).
  2. Counts images per class, reports imbalance ratio.
  3. Checks for corrupt/unreadable images (worth knowing before training).
  4. Checks image size/aspect ratio spread (informs whether your existing
     Resize(224,224) preprocessing is reasonable or whether you'll lose a lot
     of detail on some images).
  5. Creates a stratified train/val/test split (70/15/15 by default) and
     writes it to /kaggle/working/dataset_split/ as three folder trees
     (symlinks by default), ready for a training script.
  6. Saves a class-distribution bar chart to /kaggle/working/.

USAGE (in a Kaggle notebook cell):
  !python analyze_dataset.py --data_dir /kaggle/input/<your-dataset-name>

  Or, since Kaggle cells often run without convenient CLI args, you can also
  just edit DEFAULT_DATA_DIR below and run:
  !python analyze_dataset.py

If your dataset is NOT folder-per-class (e.g. flat folder + a CSV/JSON
mapping filenames to labels), pass --labels_file pointing to that CSV/JSON
and this script will use it instead of scanning folders. See --help.

If auto-detection fails or looks wrong, the script prints exactly what it
found so you can tell me and I'll adjust it to your actual structure —
this is a first pass, not a guess I'm claiming is correct.
"""

import argparse
import json
import os
import random
import shutil
import sys
from collections import Counter
from pathlib import Path

IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

# Edit this if you're running cells without CLI args.
# Check the real folder name first with: ls /kaggle/input/
DEFAULT_DATA_DIR = "/kaggle/input"
DEFAULT_OUTPUT_DIR = "/kaggle/working/dataset_split"


def find_class_folders(data_dir: Path):
    """
    Detect folder-per-class structure: data_dir/<class_name>/<image files>
    Returns dict {class_name: [file_paths]} or None if this structure isn't found.
    """
    subdirs = [d for d in data_dir.iterdir() if d.is_dir()]
    if not subdirs:
        return None

    class_map = {}
    for d in subdirs:
        images = [
            f for f in d.rglob("*")
            if f.is_file() and f.suffix.lower() in IMG_EXTENSIONS
        ]
        if images:
            class_map[d.name] = images

    # Require at least 2 "classes" with images to call this a valid folder-per-class layout
    if len(class_map) >= 2:
        return class_map
    return None


def find_labels_file(data_dir: Path, labels_file: str = None):
    """
    Detect a CSV/JSON label file mapping filename -> class label.
    Returns dict {filename: label} or None.
    """
    candidate = None
    if labels_file:
        candidate = Path(labels_file)
    else:
        # look for common names
        for name in ["labels.csv", "labels.json", "annotations.csv",
                     "train.csv", "metadata.csv", "dataset.csv"]:
            p = data_dir / name
            if p.exists():
                candidate = p
                break

    if candidate is None or not candidate.exists():
        return None

    label_map = {}
    if candidate.suffix.lower() == ".csv":
        import csv
        with open(candidate, newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            # Try to guess filename/label column names
            fieldnames = reader.fieldnames or []
            fname_col = next((c for c in fieldnames if "file" in c.lower() or "image" in c.lower() or "name" in c.lower()), None)
            label_col = next((c for c in fieldnames if "label" in c.lower() or "class" in c.lower() or "diagnosis" in c.lower()), None)
            if not fname_col or not label_col:
                print(f"[WARN] Could not confidently identify filename/label columns in {candidate}.")
                print(f"       Columns found: {fieldnames}")
                print("       Edit find_labels_file() to specify the correct column names.")
                return None
            for row in reader:
                label_map[row[fname_col]] = row[label_col]
    elif candidate.suffix.lower() == ".json":
        with open(candidate, encoding="utf-8") as f:
            raw = json.load(f)
        if isinstance(raw, dict):
            label_map = raw
        elif isinstance(raw, list):
            # assume list of {"filename":..., "label":...} dicts
            for item in raw:
                fname = item.get("filename") or item.get("image") or item.get("file")
                label = item.get("label") or item.get("class") or item.get("diagnosis")
                if fname and label:
                    label_map[fname] = label

    return label_map if label_map else None


def check_image_integrity(file_paths):
    """Returns (good_files, bad_files) — flags unreadable/corrupt images."""
    try:
        from PIL import Image
    except ImportError:
        print("[WARN] Pillow not installed — skipping image integrity check.")
        return file_paths, []

    good, bad = [], []
    for fp in file_paths:
        try:
            with Image.open(fp) as img:
                img.verify()
            good.append(fp)
        except Exception as e:
            bad.append((fp, str(e)))
    return good, bad


def get_image_sizes(file_paths, sample_n=200):
    """Sample up to sample_n images and report width/height stats."""
    try:
        from PIL import Image
    except ImportError:
        return []
    sample = random.sample(file_paths, min(sample_n, len(file_paths)))
    sizes = []
    for fp in sample:
        try:
            with Image.open(fp) as img:
                sizes.append(img.size)  # (width, height)
        except Exception:
            continue
    return sizes


def stratified_split(class_map, train_ratio=0.7, val_ratio=0.15, seed=42):
    """
    class_map: {class_name: [file_paths]}
    Returns {"train": {...}, "val": {...}, "test": {...}} with same structure.
    """
    random.seed(seed)
    splits = {"train": {}, "val": {}, "test": {}}
    for cls, files in class_map.items():
        files = files[:]  # copy
        random.shuffle(files)
        n = len(files)
        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)
        splits["train"][cls] = files[:n_train]
        splits["val"][cls] = files[n_train:n_train + n_val]
        splits["test"][cls] = files[n_train + n_val:]
    return splits


def write_split_to_disk(splits, output_dir: Path, copy_files=True):
    """
    Materializes the split as output_dir/{train,val,test}/<class>/<image files>.
    copy_files=True copies (safer, uses more disk); set False to symlink instead.
    """
    for split_name, class_map in splits.items():
        for cls, files in class_map.items():
            dest_dir = output_dir / split_name / cls
            dest_dir.mkdir(parents=True, exist_ok=True)
            for fp in files:
                dest = dest_dir / fp.name
                if dest.exists():
                    continue
                if copy_files:
                    shutil.copy2(fp, dest)
                else:
                    try:
                        dest.symlink_to(fp.resolve())
                    except OSError:
                        shutil.copy2(fp, dest)  # fallback on filesystems without symlink support


def plot_distribution(counts: dict, output_path: Path):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        print("[WARN] matplotlib not installed — skipping distribution plot. "
              "Install with: pip install matplotlib --break-system-packages")
        return

    classes = list(counts.keys())
    values = [counts[c] for c in classes]
    # sort descending for readability
    order = sorted(range(len(classes)), key=lambda i: -values[i])
    classes = [classes[i] for i in order]
    values = [values[i] for i in order]

    plt.figure(figsize=(10, 6))
    bars = plt.bar(range(len(classes)), values)
    plt.xticks(range(len(classes)), classes, rotation=45, ha="right")
    plt.ylabel("Image count")
    plt.title("Class distribution")
    plt.tight_layout()

    mean_count = sum(values) / len(values)
    plt.axhline(mean_count, color="red", linestyle="--", linewidth=1, label=f"mean ({mean_count:.0f})")
    plt.legend()

    plt.savefig(output_path, dpi=150)
    plt.close()
    print(f"[OK] Saved class distribution plot -> {output_path}")


def main():
    parser = argparse.ArgumentParser(description="Analyze fundus dataset before training.")
    parser.add_argument("--data_dir", default=DEFAULT_DATA_DIR,
                         help=f"Path to the dataset root (default: {DEFAULT_DATA_DIR}). "
                              f"On Kaggle this is usually /kaggle/input/<dataset-name>/ — "
                              f"run 'ls /kaggle/input/' in a cell to check the exact name.")
    parser.add_argument("--labels_file", default=None,
                         help="Optional CSV/JSON label file, if dataset is NOT folder-per-class.")
    parser.add_argument("--output_dir", default=DEFAULT_OUTPUT_DIR,
                         help=f"Where to write the train/val/test split (default: {DEFAULT_OUTPUT_DIR})")
    parser.add_argument("--train_ratio", type=float, default=0.70)
    parser.add_argument("--val_ratio", type=float, default=0.15)
    parser.add_argument("--skip_integrity_check", action="store_true",
                         help="Skip per-image corruption check (faster, less safe).")
    parser.add_argument("--copy", action="store_true",
                         help="Copy files into split dirs instead of symlinking. "
                              "Default is symlink (recommended on Kaggle — /kaggle/input/ "
                              "is read-only and copying 1.5GB wastes your working-dir quota).")
    parser.add_argument("--seed", type=int, default=42)
    # Use parse_known_args so this doesn't crash if Kaggle/Jupyter injects its own
    # kernel args (e.g. -f /path/to/kernel.json) when run via %run or similar.
    args, _unknown = parser.parse_known_args()

    data_dir = Path(args.data_dir)
    if not data_dir.exists():
        print(f"[ERROR] data_dir does not exist: {data_dir}")
        if str(data_dir) == DEFAULT_DATA_DIR:
            print("  You're using the default /kaggle/input — that's the parent folder "
                  "for ALL attached datasets, not your dataset itself.")
            print("  Run this in a cell first to find your dataset's exact folder name:")
            print("    import os; print(os.listdir('/kaggle/input'))")
            print("  Then pass it explicitly, e.g.:")
            print("    !python analyze_dataset.py --data_dir /kaggle/input/your-dataset-name")
        sys.exit(1)

    print(f"Scanning {data_dir} ...")

    # --- Step 1: detect structure ---
    class_map = find_class_folders(data_dir)
    structure_type = "folder-per-class"

    if class_map is None:
        label_map = find_labels_file(data_dir, args.labels_file)
        if label_map is None:
            print("[ERROR] Could not auto-detect dataset structure.")
            print("  Tried: folder-per-class layout, and common label-file names")
            print("  (labels.csv, labels.json, annotations.csv, train.csv, metadata.csv, dataset.csv).")
            print("  Top-level contents of data_dir:")
            for item in sorted(data_dir.iterdir())[:30]:
                print(f"    {item.name}")
            print("\n  Re-run with --labels_file pointing at your label file, "
                  "or tell me the actual structure and I'll adjust the script.")
            sys.exit(1)

        structure_type = "label-file"
        # build class_map from flat folder + label_map
        all_images = {f.name: f for f in data_dir.rglob("*")
                      if f.is_file() and f.suffix.lower() in IMG_EXTENSIONS}
        class_map = {}
        unmatched = 0
        for fname, label in label_map.items():
            fp = all_images.get(fname) or all_images.get(Path(fname).name)
            if fp is None:
                unmatched += 1
                continue
            class_map.setdefault(str(label), []).append(fp)
        if unmatched:
            print(f"[WARN] {unmatched} filenames in the label file had no matching image on disk.")

    print(f"\nDetected structure: {structure_type}")
    print(f"Classes found: {len(class_map)}")

    if len(class_map) != 10:
        print(f"[NOTE] Expected 10 classes for this dataset, found {len(class_map)}. "
              f"Double check this is the right dataset / structure before proceeding.")

    # --- Step 2: class distribution ---
    counts = {cls: len(files) for cls, files in class_map.items()}
    total = sum(counts.values())
    print(f"\nTotal images: {total}")
    print("\nPer-class counts:")
    for cls, n in sorted(counts.items(), key=lambda kv: -kv[1]):
        pct = 100 * n / total if total else 0
        print(f"  {cls:40s} {n:6d}  ({pct:5.1f}%)")

    max_c, min_c = max(counts.values()), min(counts.values())
    imbalance_ratio = max_c / min_c if min_c > 0 else float("inf")
    print(f"\nImbalance ratio (largest/smallest class): {imbalance_ratio:.1f}x")
    if imbalance_ratio >= 3:
        print("  -> Significant imbalance. Plan to use weighted loss and/or "
              "oversampling of minority classes during training.")
    elif imbalance_ratio >= 1.5:
        print("  -> Mild imbalance. Weighted loss recommended, oversampling optional.")
    else:
        print("  -> Roughly balanced. Standard cross-entropy loss should be fine.")

    # --- Step 3: image integrity check ---
    all_files = [fp for files in class_map.values() for fp in files]
    if not args.skip_integrity_check:
        print(f"\nChecking image integrity ({len(all_files)} files, this may take a moment)...")
        good_files, bad_files = check_image_integrity(all_files)
        if bad_files:
            print(f"[WARN] {len(bad_files)} corrupt/unreadable images found:")
            for fp, err in bad_files[:10]:
                print(f"    {fp}: {err}")
            if len(bad_files) > 10:
                print(f"    ... and {len(bad_files) - 10} more")
            print("  These will be excluded from the split.")
            bad_set = {fp for fp, _ in bad_files}
            class_map = {cls: [f for f in files if f not in bad_set]
                         for cls, files in class_map.items()}
        else:
            print("[OK] No corrupt images found.")
    else:
        print("\nSkipping integrity check (--skip_integrity_check).")

    # --- Step 4: image size distribution ---
    print("\nSampling image dimensions...")
    sizes = get_image_sizes(all_files)
    if sizes:
        widths = [w for w, h in sizes]
        heights = [h for w, h in sizes]
        print(f"  Width  range: {min(widths)}-{max(widths)}  (mean {sum(widths)/len(widths):.0f})")
        print(f"  Height range: {min(heights)}-{max(heights)}  (mean {sum(heights)/len(heights):.0f})")
        print("  (Existing pipeline resizes everything to 224x224 — large variance here "
              "just means more aggressive resizing, not a blocker.)")

    # --- Step 5: stratified split ---
    print(f"\nCreating stratified split (train={args.train_ratio}, val={args.val_ratio}, "
          f"test={1 - args.train_ratio - args.val_ratio:.2f})...")
    splits = stratified_split(class_map, args.train_ratio, args.val_ratio, seed=args.seed)
    for split_name, cmap in splits.items():
        n = sum(len(files) for files in cmap.values())
        print(f"  {split_name}: {n} images")
        for cls, files in sorted(cmap.items()):
            if len(files) == 0:
                print(f"    [WARN] class '{cls}' has 0 images in {split_name} split — "
                      f"too few samples in this class overall.")

    output_dir = Path(args.output_dir)
    output_dir.parent.mkdir(parents=True, exist_ok=True)
    print(f"\nWriting split to {output_dir} ({'copies' if args.copy else 'symlinks'})...")
    write_split_to_disk(splits, output_dir, copy_files=args.copy)
    print("[OK] Split written.")

    # --- Step 6: plot ---
    plot_path = Path("/kaggle/working/class_distribution.png") if Path("/kaggle/working").exists() \
        else Path("class_distribution.png")
    plot_distribution(counts, plot_path)

    # --- Step 7: save summary JSON for later reference ---
    summary = {
        "structure_type": structure_type,
        "num_classes": len(class_map),
        "total_images": total,
        "per_class_counts": counts,
        "imbalance_ratio": imbalance_ratio,
        "split_sizes": {name: sum(len(f) for f in cmap.values()) for name, cmap in splits.items()},
    }
    summary_path = Path("/kaggle/working/dataset_summary.json") if Path("/kaggle/working").exists() \
        else Path("dataset_summary.json")
    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)
    print(f"[OK] Saved {summary_path}")

    print("\nDone. Next: use the split at "
          f"{output_dir}/{{train,val,test}}/<class>/ to build train.py's data loaders.")


if __name__ == "__main__":
    main()

Scanning /kaggle/input ...
[ERROR] Could not auto-detect dataset structure.
  Tried: folder-per-class layout, and common label-file names
  (labels.csv, labels.json, annotations.csv, train.csv, metadata.csv, dataset.csv).
  Top-level contents of data_dir:
    datasets

  Re-run with --labels_file pointing at your label file, or tell me the actual structure and I'll adjust the script.


SystemExit: 1

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [4]:
import sys
sys.argv = [
    "analyze_dataset.py",
    "--data_dir", "/kaggle/input/datasets/ayush3954/fundus-images/Original Dataset/Original Dataset"
]
main()

Scanning /kaggle/input/datasets/ayush3954/fundus-images/Original Dataset/Original Dataset ...

Detected structure: folder-per-class
Classes found: 10

Total images: 5335

Per-class counts:
  Diabetic Retinopathy                       1509  ( 28.3%)
  Glaucoma                                   1349  ( 25.3%)
  Healthy                                    1024  ( 19.2%)
  Myopia                                      500  (  9.4%)
  Macular Scar                                444  (  8.3%)
  Retinitis Pigmentosa                        139  (  2.6%)
  Disc Edema                                  127  (  2.4%)
  Retinal Detachment                          125  (  2.3%)
  Central Serous Chorioretinopathy            101  (  1.9%)
  Pterygium                                    17  (  0.3%)

Imbalance ratio (largest/smallest class): 88.8x
  -> Significant imbalance. Plan to use weighted loss and/or oversampling of minority classes during training.

Checking image integrity (5335 files, this may tak

In [5]:
import os
print(os.path.exists('/kaggle/working/dataset_split'))
print(os.listdir('/kaggle/working/dataset_split')) if os.path.exists('/kaggle/working/dataset_split') else None

True
['val', 'test', 'train']


In [6]:
"""
train.py  (Kaggle Notebook version)

Day 2-3 script: fine-tune EfficientNet-B0 (and, with --arch mobilenet_v2,
MobileNet) on the 10-class fundus dataset produced by analyze_dataset.py.

WHY THIS DESIGN (worth remembering for interviews):
  Your dataset has an 88.8x class imbalance — the smallest class
  (Pterygium) has only ~12 training images after the 70/15/15 split.
  With that little data:
    - The backbone is FROZEN by default. More trainable parameters with
      that few minority-class samples overfits fast, especially on the
      classes you can least afford to overfit on. Only the classifier
      head is trained.
    - A WeightedRandomSampler is used (not just a weighted loss) so rare
      classes actually appear in most batches, rather than mostly being
      absent from all but a handful of steps.
    - A CLASS-WEIGHTED loss (inverse-sqrt-frequency, not raw inverse
      frequency — raw inverse frequency lets an 88x ratio dominate
      gradients and destabilizes training) provides a second layer of
      correction on top of the sampler.
    - Checkpointing is done on VALIDATION MACRO-F1, not accuracy. With
      this imbalance, accuracy alone is maximized by ignoring minority
      classes entirely (the model could get ~75% accuracy by only ever
      predicting the top 3 classes). Macro-F1 forces genuine attempt on
      every class.
    - --unfreeze_last_block is provided as an OPT-IN experiment, not the
      default. Run both configs and compare macro-F1 — that comparison
      is the actual deliverable, not just picking one.

KAGGLE NOTES:
  - Assumes the split produced by analyze_dataset.py at
    /kaggle/working/dataset_split/{train,val,test}/<class>/
  - Checkpoints and logs are written to /kaggle/working/ (the only
    writable directory in a Kaggle session).
  - If your Kaggle notebook has "Accelerator: GPU" enabled, this will
    use CUDA automatically. If training is slow, check
    Settings -> Accelerator in the notebook sidebar.

USAGE (paste into a cell, then in the NEXT cell call main() the same way
you did for analyze_dataset.py, since !python won't see a pasted-in-cell
file):

    import sys
    sys.argv = [
        "train.py",
        "--arch", "efficientnet_b0",
        "--epochs", "15",
    ]
    main()

Run once with efficientnet_b0, once with mobilenet_v2 (Day 4 in the plan),
so both checkpoints exist for the ensemble step later. To run the
last-block-unfrozen experiment for comparison:

    sys.argv = ["train.py", "--arch", "efficientnet_b0", "--epochs", "15",
                "--unfreeze_last_block", "--run_name", "efficientnet_unfrozen"]
    main()
"""

import argparse
import json
import sys
import time
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, models, transforms

DEFAULT_SPLIT_DIR = "/kaggle/working/dataset_split"
DEFAULT_OUTPUT_DIR = "/kaggle/working"

IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".ppm", ".pgm", ".webp"}
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------

def build_transforms(train: bool):
    """
    Matches the existing predictor.py pipeline (ImageNet normalization,
    224x224) but adds augmentation for the train split — important here
    since minority classes have so few real images that augmentation is
    doing real work, not just marginal regularization.

    NOTE: this does NOT include the project's existing CLAHE step
    (_apply_clahe in predictor.py) because that's an inference-time
    preprocessing choice tied to the deployed pipeline, not training
    augmentation. If your original models were trained WITH CLAHE
    applied first, add it here too via a Lambda transform, matching
    predictor.py's _apply_clahe exactly — otherwise train/inference
    preprocessing will silently mismatch, which would quietly hurt
    accuracy in a way that's easy to miss.
    """
    if train:
        return transforms.Compose([
            transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])


def build_weighted_sampler(dataset: datasets.ImageFolder):
    """
    WeightedRandomSampler so minority classes (down to ~12 images) show up
    in most batches instead of being nearly absent from training. Sample
    weight per example = 1 / class_count, so each class contributes roughly
    equally in expectation over an epoch.
    """
    targets = [label for _, label in dataset.samples]
    class_counts = torch.bincount(torch.tensor(targets))
    class_weights = 1.0 / class_counts.float()
    sample_weights = class_weights[torch.tensor(targets)]
    return WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
    )


def build_loss_class_weights(dataset: datasets.ImageFolder, device):
    """
    Inverse-SQRT-frequency class weights for the loss function.
    Plain inverse frequency (1/count) would let the 88.8x imbalance ratio
    dominate the loss directly on top of the sampler already correcting
    exposure — sqrt softens that so both mechanisms don't compound into
    instability (rare-class gradients swamping common-class learning).
    """
    targets = [label for _, label in dataset.samples]
    class_counts = torch.bincount(torch.tensor(targets)).float()
    weights = 1.0 / torch.sqrt(class_counts)
    weights = weights / weights.sum() * len(class_counts)  # normalize, mean weight ~1
    return weights.to(device)


def safe_image_folder(root: str, transform):
    """
    Wraps datasets.ImageFolder with a guard against empty class subfolders.
    Plain ImageFolder crashes with FileNotFoundError if ANY class folder is
    empty (e.g. a rare class that got 0 images in the val/test split) —
    a real risk here given classes as small as ~12 total images. This
    removes empty class dirs from consideration before construction, and
    reports which classes were dropped so it's never a silent data loss.
    """
    root_path = Path(root)
    empty_classes = []
    for d in sorted(root_path.iterdir()):
        if d.is_dir():
            has_images = any(
                f.is_file() and f.suffix.lower() in IMG_EXTENSIONS
                for f in d.rglob("*")
            )
            if not has_images:
                empty_classes.append(d.name)

    if empty_classes:
        print(f"[WARN] Empty class folder(s) in {root}: {empty_classes}. "
              f"These classes have 0 images in this split and will be excluded "
              f"from THIS split's dataset object — this is a symptom of severe "
              f"class imbalance (some classes are too small to appear in every "
              f"split), not something this script can fix. If this is your val "
              f"or test split, that class's metrics will be reported as "
              f"'no data' rather than silently skipped.")

    def is_valid_file(path):
        return True  # extension filtering handled by ImageFolder itself

    dataset = datasets.ImageFolder(
        root,
        transform=transform,
        is_valid_file=lambda p: Path(p).suffix.lower() in IMG_EXTENSIONS,
    ) if not empty_classes else _image_folder_excluding(root, transform, empty_classes)

    return dataset, empty_classes


def _image_folder_excluding(root: str, transform, exclude_classes):
    """
    Builds an ImageFolder-equivalent dataset while excluding specific class
    subfolders entirely (rather than letting them raise on construction).
    Necessary because torchvision's ImageFolder does not support excluding
    a subset of discovered classes directly.
    """
    root_path = Path(root)
    tmp_classes = [d.name for d in sorted(root_path.iterdir())
                    if d.is_dir() and d.name not in exclude_classes]
    if not tmp_classes:
        raise RuntimeError(f"No non-empty class folders found under {root} "
                            f"after excluding {exclude_classes}.")

    # Build directly rather than via a temp-dir copy, so this works whether
    # the split was written with symlinks or real copies.
    class_to_idx = {cls: i for i, cls in enumerate(tmp_classes)}
    samples = []
    for cls in tmp_classes:
        for f in sorted((root_path / cls).rglob("*")):
            if f.is_file() and f.suffix.lower() in IMG_EXTENSIONS:
                samples.append((str(f), class_to_idx[cls]))

    ds = datasets.ImageFolder.__new__(datasets.ImageFolder)
    ds.root = str(root_path)
    ds.transform = transform
    ds.target_transform = None
    ds.loader = datasets.folder.default_loader
    ds.extensions = tuple(IMG_EXTENSIONS)
    ds.classes = tmp_classes
    ds.class_to_idx = class_to_idx
    ds.samples = samples
    ds.targets = [s[1] for s in samples]
    ds.imgs = ds.samples
    return ds


# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------

def build_model(arch: str, num_classes: int, unfreeze_last_block: bool):
    if arch == "efficientnet_b0":
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        # Freeze everything first
        for p in model.parameters():
            p.requires_grad = False
        # Replace head (matches existing predictor.py: classifier[1] -> Linear(1280, N))
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
        for p in model.classifier.parameters():
            p.requires_grad = True
        if unfreeze_last_block:
            # model.features[-1] is EfficientNet-B0's last conv block —
            # same layer already used as the Grad-CAM target in predictor.py.
            for p in model.features[-1].parameters():
                p.requires_grad = True

    elif arch == "mobilenet_v2":
        model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
        for p in model.parameters():
            p.requires_grad = False
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
        for p in model.classifier.parameters():
            p.requires_grad = True
        if unfreeze_last_block:
            for p in model.features[-1].parameters():
                p.requires_grad = True

    else:
        raise ValueError(f"Unsupported --arch: {arch}. Use efficientnet_b0 or mobilenet_v2.")

    return model


# ---------------------------------------------------------------------------
# Train / eval loops
# ---------------------------------------------------------------------------

def run_epoch(model, loader, criterion, optimizer, device, train: bool, num_classes: int):
    model.train() if train else model.eval()

    total_loss = 0.0
    all_preds, all_targets = [], []

    torch.set_grad_enabled(train)
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)

        if train:
            optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, targets)

        if train:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(1)
        all_preds.extend(preds.cpu().tolist())
        all_targets.extend(targets.cpu().tolist())
    torch.set_grad_enabled(True)

    avg_loss = total_loss / len(loader.dataset)
    macro_f1, per_class = compute_macro_f1(all_preds, all_targets, num_classes)
    accuracy = sum(p == t for p, t in zip(all_preds, all_targets)) / len(all_targets)

    return avg_loss, accuracy, macro_f1, per_class


def compute_macro_f1(preds, targets, num_classes):
    """
    Per-class precision/recall/F1 + macro-F1, computed without sklearn
    (keeps this script dependency-light — sklearn is usually present on
    Kaggle, but this avoids relying on it just for this).
    Returns (macro_f1, per_class_dict) where per_class_dict maps
    class_index -> {precision, recall, f1, support}.
    """
    per_class = {}
    f1_scores = []
    for c in range(num_classes):
        tp = sum(1 for p, t in zip(preds, targets) if p == c and t == c)
        fp = sum(1 for p, t in zip(preds, targets) if p == c and t != c)
        fn = sum(1 for p, t in zip(preds, targets) if p != c and t == c)
        support = sum(1 for t in targets if t == c)

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

        per_class[c] = {"precision": precision, "recall": recall, "f1": f1, "support": support}
        f1_scores.append(f1)

    macro_f1 = sum(f1_scores) / len(f1_scores) if f1_scores else 0.0
    return macro_f1, per_class


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    parser = argparse.ArgumentParser(description="Fine-tune EfficientNet-B0/MobileNet on the 10-class fundus dataset.")
    parser.add_argument("--split_dir", default=DEFAULT_SPLIT_DIR,
                         help=f"Path to the {{train,val,test}} split (default: {DEFAULT_SPLIT_DIR})")
    parser.add_argument("--output_dir", default=DEFAULT_OUTPUT_DIR)
    parser.add_argument("--arch", default="efficientnet_b0",
                         choices=["efficientnet_b0", "mobilenet_v2"],
                         help="mobilenet_v2 is torchvision's closest match to the project's "
                              "existing mobilenet_best.pth naming; adjust if your original "
                              "MobileNet was a different variant.")
    parser.add_argument("--epochs", type=int, default=15)
    parser.add_argument("--batch_size", type=int, default=32)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--unfreeze_last_block", action="store_true",
                         help="Opt-in: also fine-tune the last conv block, not just the "
                              "classifier head. Run with and without this flag and compare "
                              "val macro-F1 — that comparison is the point, not just picking one.")
    parser.add_argument("--run_name", default=None,
                         help="Name for this run's checkpoint/log files. Defaults to --arch "
                              "(+ '_unfrozen' if --unfreeze_last_block).")
    parser.add_argument("--num_workers", type=int, default=2)
    parser.add_argument("--seed", type=int, default=42)
    args, _unknown = parser.parse_known_args()

    torch.manual_seed(args.seed)

    run_name = args.run_name or (args.arch + ("_unfrozen" if args.unfreeze_last_block else ""))
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    if device.type == "cpu":
        print("[WARN] No GPU detected. If you enabled a Kaggle GPU accelerator, "
              "check Settings -> Accelerator in the notebook sidebar — training "
              "on CPU will be substantially slower.")

    split_dir = Path(args.split_dir)
    train_dir, val_dir = split_dir / "train", split_dir / "val"
    if not train_dir.exists() or not val_dir.exists():
        print(f"[ERROR] Expected {train_dir} and {val_dir} to exist. "
              f"Run analyze_dataset.py first to create the split.")
        sys.exit(1)

    # --- Data ---
    train_dataset, train_empty = safe_image_folder(str(train_dir), build_transforms(train=True))
    val_dataset, val_empty = safe_image_folder(str(val_dir), build_transforms(train=False))

    # ImageFolder sorts class names alphabetically to assign indices — save
    # this mapping now so eval_baseline.py and predictor.py stay consistent
    # with what this run actually trained on.
    class_to_idx = train_dataset.class_to_idx
    idx_to_class = {v: k for k, v in class_to_idx.items()}
    num_classes = len(class_to_idx)
    print(f"Classes ({num_classes}): {class_to_idx}")

    if val_dataset.class_to_idx != class_to_idx:
        print("[ERROR] train/val have a DIFFERENT set of non-empty classes "
              f"(train excluded: {train_empty}, val excluded: {val_empty}). "
              "This means the model's output layer (sized from train classes) "
              "won't align with val's class indices, and val metrics for the "
              "mismatched classes would be meaningless if training proceeded. "
              "This is a direct consequence of severe class imbalance — some "
              "classes are too small to have any images in every split. "
              "Fix by either: (a) reducing val_ratio/test_ratio so more of a "
              "rare class's images stay in train, or (b) accepting those "
              "classes cannot be validated and excluding them from num_classes "
              "consistently across train/val/test — this script does not "
              "silently choose (b) for you.")
        sys.exit(1)

    sampler = build_weighted_sampler(train_dataset)
    train_loader = DataLoader(train_dataset, batch_size=args.batch_size,
                               sampler=sampler, num_workers=args.num_workers)
    val_loader = DataLoader(val_dataset, batch_size=args.batch_size,
                             shuffle=False, num_workers=args.num_workers)

    class_weights = build_loss_class_weights(train_dataset, device)
    print(f"Loss class weights (by class index): {[round(w, 3) for w in class_weights.cpu().tolist()]}")

    # --- Model ---
    model = build_model(args.arch, num_classes, args.unfreeze_last_block).to(device)
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable params: {trainable_params:,} / {total_params:,} "
          f"({100*trainable_params/total_params:.1f}%)")

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad], lr=args.lr
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=2
    )

    # --- Train loop ---
    best_val_macro_f1 = -1.0
    history = []
    ckpt_path = output_dir / f"{run_name}_best.pth"
    log_path = output_dir / f"{run_name}_training_log.json"

    print(f"\nTraining run: {run_name}")
    print(f"Checkpoint will be saved to: {ckpt_path}\n")

    for epoch in range(1, args.epochs + 1):
        t0 = time.time()
        train_loss, train_acc, train_f1, _ = run_epoch(
            model, train_loader, criterion, optimizer, device, train=True, num_classes=num_classes
        )
        val_loss, val_acc, val_f1, val_per_class = run_epoch(
            model, val_loader, criterion, optimizer, device, train=False, num_classes=num_classes
        )
        scheduler.step(val_f1)
        elapsed = time.time() - t0

        print(f"Epoch {epoch:3d}/{args.epochs}  "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.3f} train_f1={train_f1:.3f}  |  "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.3f} val_f1={val_f1:.3f}  "
              f"({elapsed:.1f}s)")

        history.append({
            "epoch": epoch, "train_loss": train_loss, "train_acc": train_acc, "train_f1": train_f1,
            "val_loss": val_loss, "val_acc": val_acc, "val_f1": val_f1,
        })

        if val_f1 > best_val_macro_f1:
            best_val_macro_f1 = val_f1
            torch.save({
                "model_state_dict": model.state_dict(),
                "arch": args.arch,
                "num_classes": num_classes,
                "class_to_idx": class_to_idx,
                "idx_to_class": idx_to_class,
                "unfreeze_last_block": args.unfreeze_last_block,
                "val_macro_f1": val_f1,
                "val_accuracy": val_acc,
                "epoch": epoch,
            }, ckpt_path)
            print(f"  -> New best (val_macro_f1={val_f1:.3f}). Saved to {ckpt_path}")

    # --- Final report: per-class breakdown WITH support counts, so it's ---
    # --- immediately visible which classes have statistically meaningful ---
    # --- numbers vs which don't (per the "report honestly" decision). ---
    print(f"\n{'='*70}")
    print(f"Best val macro-F1: {best_val_macro_f1:.3f} (epoch {history[-1]['epoch']})")
    print(f"\nFinal-epoch per-class validation breakdown:")
    print(f"{'Class':40s} {'Precision':>10s} {'Recall':>8s} {'F1':>8s} {'Support':>8s}  Note")
    for idx in sorted(val_per_class.keys()):
        cls_name = idx_to_class[idx]
        m = val_per_class[idx]
        note = ""
        if m["support"] < 10:
            note = "<- too few val samples to trust this number"
        elif m["support"] < 25:
            note = "<- low sample count, treat with caution"
        print(f"{cls_name:40s} {m['precision']:10.3f} {m['recall']:8.3f} {m['f1']:8.3f} "
              f"{m['support']:8d}  {note}")

    with open(log_path, "w") as f:
        json.dump({
            "run_name": run_name, "arch": args.arch,
            "unfreeze_last_block": args.unfreeze_last_block,
            "best_val_macro_f1": best_val_macro_f1,
            "class_to_idx": class_to_idx,
            "history": history,
            "final_val_per_class": {idx_to_class[k]: v for k, v in val_per_class.items()},
        }, f, indent=2)
    print(f"\n[OK] Saved training log -> {log_path}")
    print(f"[OK] Best checkpoint -> {ckpt_path}")
    print(f"\nNext: run this same script with --arch mobilenet_v2 for the ensemble step, "
          f"then eval_baseline.py against the held-out test split.")


if __name__ == "__main__":
    main()

Device: cuda
Classes (10): {'Central Serous Chorioretinopathy': 0, 'Diabetic Retinopathy': 1, 'Disc Edema': 2, 'Glaucoma': 3, 'Healthy': 4, 'Macular Scar': 5, 'Myopia': 6, 'Pterygium': 7, 'Retinal Detachment': 8, 'Retinitis Pigmentosa': 9}
Loss class weights (by class index): [1.262, 0.325, 1.125, 0.344, 0.394, 0.6, 0.564, 3.183, 1.132, 1.072]
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 140MB/s]


Trainable params: 12,810 / 4,020,358 (0.3%)

Training run: efficientnet_b0
Checkpoint will be saved to: /kaggle/working/efficientnet_b0_best.pth

Epoch   1/15  train_loss=1.1174 train_acc=0.446 train_f1=0.380  |  val_loss=1.8571 val_acc=0.269 val_f1=0.328  (119.2s)
  -> New best (val_macro_f1=0.328). Saved to /kaggle/working/efficientnet_b0_best.pth


KeyboardInterrupt: 

In [7]:
   import sys
   sys.argv = ["train.py", "--arch", "efficientnet_b0", "--epochs", "15"]
   main()

Device: cuda
Classes (10): {'Central Serous Chorioretinopathy': 0, 'Diabetic Retinopathy': 1, 'Disc Edema': 2, 'Glaucoma': 3, 'Healthy': 4, 'Macular Scar': 5, 'Myopia': 6, 'Pterygium': 7, 'Retinal Detachment': 8, 'Retinitis Pigmentosa': 9}
Loss class weights (by class index): [1.262, 0.325, 1.125, 0.344, 0.394, 0.6, 0.564, 3.183, 1.132, 1.072]
Trainable params: 12,810 / 4,020,358 (0.3%)

Training run: efficientnet_b0
Checkpoint will be saved to: /kaggle/working/efficientnet_b0_best.pth

Epoch   1/15  train_loss=1.1174 train_acc=0.446 train_f1=0.380  |  val_loss=1.8571 val_acc=0.269 val_f1=0.328  (119.0s)
  -> New best (val_macro_f1=0.328). Saved to /kaggle/working/efficientnet_b0_best.pth
Epoch   2/15  train_loss=0.7218 train_acc=0.598 train_f1=0.548  |  val_loss=1.6151 val_acc=0.361 val_f1=0.398  (118.7s)
  -> New best (val_macro_f1=0.398). Saved to /kaggle/working/efficientnet_b0_best.pth
Epoch   3/15  train_loss=0.5995 train_acc=0.645 train_f1=0.601  |  val_loss=1.6066 val_acc=0.376

In [8]:
import sys
sys.argv = [
    "train.py",
    "--arch", "efficientnet_b0",
    "--epochs", "10",
    "--unfreeze_last_block",
    "--run_name", "efficientnet_unfrozen",
]
main()

Device: cuda
Classes (10): {'Central Serous Chorioretinopathy': 0, 'Diabetic Retinopathy': 1, 'Disc Edema': 2, 'Glaucoma': 3, 'Healthy': 4, 'Macular Scar': 5, 'Myopia': 6, 'Pterygium': 7, 'Retinal Detachment': 8, 'Retinitis Pigmentosa': 9}
Loss class weights (by class index): [1.262, 0.325, 1.125, 0.344, 0.394, 0.6, 0.564, 3.183, 1.132, 1.072]
Trainable params: 424,970 / 4,020,358 (10.6%)

Training run: efficientnet_unfrozen
Checkpoint will be saved to: /kaggle/working/efficientnet_unfrozen_best.pth

Epoch   1/10  train_loss=0.8260 train_acc=0.543 train_f1=0.503  |  val_loss=1.4820 val_acc=0.430 val_f1=0.465  (120.6s)
  -> New best (val_macro_f1=0.465). Saved to /kaggle/working/efficientnet_unfrozen_best.pth
Epoch   2/10  train_loss=0.4705 train_acc=0.702 train_f1=0.677  |  val_loss=1.2121 val_acc=0.546 val_f1=0.561  (119.8s)
  -> New best (val_macro_f1=0.561). Saved to /kaggle/working/efficientnet_unfrozen_best.pth
Epoch   3/10  train_loss=0.3667 train_acc=0.757 train_f1=0.732  |  val

In [9]:
import sys
sys.argv = [
    "train.py",
    "--arch", "mobilenet_v2",
    "--epochs", "10",
    "--unfreeze_last_block",
    "--run_name", "mobilenet_unfrozen",
]
main()

Device: cuda
Classes (10): {'Central Serous Chorioretinopathy': 0, 'Diabetic Retinopathy': 1, 'Disc Edema': 2, 'Glaucoma': 3, 'Healthy': 4, 'Macular Scar': 5, 'Myopia': 6, 'Pterygium': 7, 'Retinal Detachment': 8, 'Retinitis Pigmentosa': 9}
Loss class weights (by class index): [1.262, 0.325, 1.125, 0.344, 0.394, 0.6, 0.564, 3.183, 1.132, 1.072]
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 106MB/s] 

Trainable params: 424,970 / 2,236,682 (19.0%)

Training run: mobilenet_unfrozen
Checkpoint will be saved to: /kaggle/working/mobilenet_unfrozen_best.pth



Epoch   1/10  train_loss=0.6805 train_acc=0.610 train_f1=0.582  |  val_loss=1.2201 val_acc=0.544 val_f1=0.555  (120.0s)
  -> New best (val_macro_f1=0.555). Saved to /kaggle/working/mobilenet_unfrozen_best.pth
Epoch   2/10  train_loss=0.4559 train_acc=0.705 train_f1=0.691  |  val_loss=1.1965 val_acc=0.584 val_f1=0.574  (111.2s)
  -> New best (val_macro_f1=0.574). Saved to /kaggle/working/mobilenet_unfrozen_best.pth
Epoch   3/10  train_loss=0.4425 train_acc=0.728 train_f1=0.720  |  val_loss=1.1344 val_acc=0.589 val_f1=0.601  (115.1s)
  -> New best (val_macro_f1=0.601). Saved to /kaggle/working/mobilenet_unfrozen_best.pth
Epoch   4/10  train_loss=0.3854 train_acc=0.756 train_f1=0.749  |  val_loss=1.3089 val_acc=0.533 val_f1=0.562  (114.5s)
Epoch   5/10  train_loss=0.3598 train_acc=0.759 train_f1=0.751  |  val_loss=1.0256 val_acc=0.623 val_f1=0.632  (113.4s)
  -> New best (val_macro_f1=0.632). Saved to /kaggle/working/mobilenet_unfrozen_best.pth
Epoch   6/10  train_loss=0.3691 train_acc=0.

In [10]:
"""
train.py  (Kaggle Notebook version)

Day 2-3 script: fine-tune EfficientNet-B0 (and, with --arch mobilenet_v2,
MobileNet) on the 10-class fundus dataset produced by analyze_dataset.py.

WHY THIS DESIGN (worth remembering for interviews):
  Your dataset has an 88.8x class imbalance — the smallest class
  (Pterygium) has only ~12 training images after the 70/15/15 split.
  With that little data:
    - The backbone is FROZEN by default. More trainable parameters with
      that few minority-class samples overfits fast, especially on the
      classes you can least afford to overfit on. Only the classifier
      head is trained.
    - A WeightedRandomSampler is used (not just a weighted loss) so rare
      classes actually appear in most batches, rather than mostly being
      absent from all but a handful of steps.
    - A CLASS-WEIGHTED loss (inverse-sqrt-frequency, not raw inverse
      frequency — raw inverse frequency lets an 88x ratio dominate
      gradients and destabilizes training) provides a second layer of
      correction on top of the sampler.
    - Checkpointing is done on VALIDATION MACRO-F1, not accuracy. With
      this imbalance, accuracy alone is maximized by ignoring minority
      classes entirely (the model could get ~75% accuracy by only ever
      predicting the top 3 classes). Macro-F1 forces genuine attempt on
      every class.
    - --unfreeze_last_block is provided as an OPT-IN experiment, not the
      default. Run both configs and compare macro-F1 — that comparison
      is the actual deliverable, not just picking one.

KAGGLE NOTES:
  - Assumes the split produced by analyze_dataset.py at
    /kaggle/working/dataset_split/{train,val,test}/<class>/
  - Checkpoints and logs are written to /kaggle/working/ (the only
    writable directory in a Kaggle session).
  - If your Kaggle notebook has "Accelerator: GPU" enabled, this will
    use CUDA automatically. If training is slow, check
    Settings -> Accelerator in the notebook sidebar.

USAGE (paste into a cell, then in the NEXT cell call main() the same way
you did for analyze_dataset.py, since !python won't see a pasted-in-cell
file):

    import sys
    sys.argv = [
        "train.py",
        "--arch", "efficientnet_b0",
        "--epochs", "15",
    ]
    main()

Run once with efficientnet_b0, once with mobilenet_v2 (Day 4 in the plan),
so both checkpoints exist for the ensemble step later. To run the
last-block-unfrozen experiment for comparison:

    sys.argv = ["train.py", "--arch", "efficientnet_b0", "--epochs", "15",
                "--unfreeze_last_block", "--run_name", "efficientnet_unfrozen"]
    main()
"""

import argparse
import json
import sys
import time
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, models, transforms

DEFAULT_SPLIT_DIR = "/kaggle/working/dataset_split"
DEFAULT_OUTPUT_DIR = "/kaggle/working"

IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".ppm", ".pgm", ".webp"}
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------

def build_transforms(train: bool, intensity: str = "standard"):
    """
    Matches the existing predictor.py pipeline (ImageNet normalization,
    224x224) but adds augmentation for the train split — important here
    since minority classes have so few real images that augmentation is
    doing real work, not just marginal regularization.

    intensity: "standard" (well-sampled classes) or "heavy" (minority
    classes). Heavy augmentation squeezes more effective variation out of
    the handful of real images available for classes like Pterygium
    (~12 train images) — standard augmentation on a class that small
    barely perturbs the tiny set of images the model sees repeatedly.

    Fundus images have no fixed "up" orientation (unlike natural photos
    of faces/text), so both horizontal AND vertical flips plus wider
    rotation ranges are safe here and would not be for most photo data.

    NOTE: this does NOT include the project's existing CLAHE step
    (_apply_clahe in predictor.py) because that's an inference-time
    preprocessing choice tied to the deployed pipeline, not training
    augmentation. If your original models were trained WITH CLAHE
    applied first, add it here too via a Lambda transform, matching
    predictor.py's _apply_clahe exactly — otherwise train/inference
    preprocessing will silently mismatch, which would quietly hurt
    accuracy in a way that's easy to miss.
    """
    if not train:
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])

    if intensity == "heavy":
        return transforms.Compose([
            transforms.RandomResizedCrop(224, scale=(0.70, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(30),
            transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.2, hue=0.02),
            transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            transforms.RandomErasing(p=0.3, scale=(0.02, 0.10)),
        ])
    else:  # standard
        return transforms.Compose([
            transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            transforms.RandomErasing(p=0.15, scale=(0.02, 0.08)),
        ])


class PerClassAugmentDataset(torch.utils.data.Dataset):
    """
    Wraps an ImageFolder-like dataset to apply a DIFFERENT transform
    depending on the sample's class — heavy augmentation for minority
    classes, standard for well-sampled ones. Plain ImageFolder can only
    apply one fixed transform to every sample, which doesn't let us target
    augmentation at the classes that actually need it (the data-scarce
    ones), so this wraps the already-loaded PIL images (transform=None on
    the base dataset) and applies transforms here based on each sample's
    label.
    """

    def __init__(self, base_dataset, heavy_class_indices: set,
                 standard_transform, heavy_transform):
        self.base = base_dataset  # must have transform=None; yields (PIL.Image, label)
        self.heavy_class_indices = heavy_class_indices
        self.standard_transform = standard_transform
        self.heavy_transform = heavy_transform

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        image, label = self.base[idx]
        transform = self.heavy_transform if label in self.heavy_class_indices else self.standard_transform
        return transform(image), label


def determine_heavy_classes(dataset, threshold: int = 100):
    """
    Returns the set of class indices with fewer than `threshold` training
    images — these get heavy augmentation. 100 is chosen to sit between
    your well-sampled classes (Diabetic Retinopathy, Glaucoma, Healthy,
    Myopia, Macular Scar — all 300+ train images) and your scarce ones
    (Retinitis Pigmentosa, Disc Edema, Retinal Detachment, CSCR, Pterygium
    — all under 100 train images), based on the real class distribution
    from analyze_dataset.py.
    """
    targets = [label for _, label in dataset.samples]
    class_counts = torch.bincount(torch.tensor(targets))
    heavy = {i for i, count in enumerate(class_counts.tolist()) if count < threshold}
    return heavy


def build_weighted_sampler(dataset: datasets.ImageFolder):
    """
    WeightedRandomSampler so minority classes (down to ~12 images) show up
    in most batches instead of being nearly absent from training. Sample
    weight per example = 1 / class_count, so each class contributes roughly
    equally in expectation over an epoch.
    """
    targets = [label for _, label in dataset.samples]
    class_counts = torch.bincount(torch.tensor(targets))
    class_weights = 1.0 / class_counts.float()
    sample_weights = class_weights[torch.tensor(targets)]
    return WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
    )


def build_loss_class_weights(dataset: datasets.ImageFolder, device):
    """
    Inverse-SQRT-frequency class weights for the loss function.
    Plain inverse frequency (1/count) would let the 88.8x imbalance ratio
    dominate the loss directly on top of the sampler already correcting
    exposure — sqrt softens that so both mechanisms don't compound into
    instability (rare-class gradients swamping common-class learning).
    """
    targets = [label for _, label in dataset.samples]
    class_counts = torch.bincount(torch.tensor(targets)).float()
    weights = 1.0 / torch.sqrt(class_counts)
    weights = weights / weights.sum() * len(class_counts)  # normalize, mean weight ~1
    return weights.to(device)


def safe_image_folder(root: str, transform):
    """
    Wraps datasets.ImageFolder with a guard against empty class subfolders.
    Plain ImageFolder crashes with FileNotFoundError if ANY class folder is
    empty (e.g. a rare class that got 0 images in the val/test split) —
    a real risk here given classes as small as ~12 total images. This
    removes empty class dirs from consideration before construction, and
    reports which classes were dropped so it's never a silent data loss.
    """
    root_path = Path(root)
    empty_classes = []
    for d in sorted(root_path.iterdir()):
        if d.is_dir():
            has_images = any(
                f.is_file() and f.suffix.lower() in IMG_EXTENSIONS
                for f in d.rglob("*")
            )
            if not has_images:
                empty_classes.append(d.name)

    if empty_classes:
        print(f"[WARN] Empty class folder(s) in {root}: {empty_classes}. "
              f"These classes have 0 images in this split and will be excluded "
              f"from THIS split's dataset object — this is a symptom of severe "
              f"class imbalance (some classes are too small to appear in every "
              f"split), not something this script can fix. If this is your val "
              f"or test split, that class's metrics will be reported as "
              f"'no data' rather than silently skipped.")

    def is_valid_file(path):
        return True  # extension filtering handled by ImageFolder itself

    dataset = datasets.ImageFolder(
        root,
        transform=transform,
        is_valid_file=lambda p: Path(p).suffix.lower() in IMG_EXTENSIONS,
    ) if not empty_classes else _image_folder_excluding(root, transform, empty_classes)

    return dataset, empty_classes


def _image_folder_excluding(root: str, transform, exclude_classes):
    """
    Builds an ImageFolder-equivalent dataset while excluding specific class
    subfolders entirely (rather than letting them raise on construction).
    Necessary because torchvision's ImageFolder does not support excluding
    a subset of discovered classes directly.
    """
    root_path = Path(root)
    tmp_classes = [d.name for d in sorted(root_path.iterdir())
                    if d.is_dir() and d.name not in exclude_classes]
    if not tmp_classes:
        raise RuntimeError(f"No non-empty class folders found under {root} "
                            f"after excluding {exclude_classes}.")

    # Build directly rather than via a temp-dir copy, so this works whether
    # the split was written with symlinks or real copies.
    class_to_idx = {cls: i for i, cls in enumerate(tmp_classes)}
    samples = []
    for cls in tmp_classes:
        for f in sorted((root_path / cls).rglob("*")):
            if f.is_file() and f.suffix.lower() in IMG_EXTENSIONS:
                samples.append((str(f), class_to_idx[cls]))

    ds = datasets.ImageFolder.__new__(datasets.ImageFolder)
    ds.root = str(root_path)
    ds.transform = transform
    ds.target_transform = None
    ds.loader = datasets.folder.default_loader
    ds.extensions = tuple(IMG_EXTENSIONS)
    ds.classes = tmp_classes
    ds.class_to_idx = class_to_idx
    ds.samples = samples
    ds.targets = [s[1] for s in samples]
    ds.imgs = ds.samples
    return ds


# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------

def build_model(arch: str, num_classes: int, unfreeze_last_block: bool):
    if arch == "efficientnet_b0":
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        # Freeze everything first
        for p in model.parameters():
            p.requires_grad = False
        # Replace head (matches existing predictor.py: classifier[1] -> Linear(1280, N))
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
        for p in model.classifier.parameters():
            p.requires_grad = True
        if unfreeze_last_block:
            # model.features[-1] is EfficientNet-B0's last conv block —
            # same layer already used as the Grad-CAM target in predictor.py.
            for p in model.features[-1].parameters():
                p.requires_grad = True

    elif arch == "mobilenet_v2":
        model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
        for p in model.parameters():
            p.requires_grad = False
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
        for p in model.classifier.parameters():
            p.requires_grad = True
        if unfreeze_last_block:
            for p in model.features[-1].parameters():
                p.requires_grad = True

    else:
        raise ValueError(f"Unsupported --arch: {arch}. Use efficientnet_b0 or mobilenet_v2.")

    return model


# ---------------------------------------------------------------------------
# Train / eval loops
# ---------------------------------------------------------------------------

def run_epoch(model, loader, criterion, optimizer, device, train: bool, num_classes: int):
    model.train() if train else model.eval()

    total_loss = 0.0
    all_preds, all_targets = [], []

    torch.set_grad_enabled(train)
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)

        if train:
            optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, targets)

        if train:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(1)
        all_preds.extend(preds.cpu().tolist())
        all_targets.extend(targets.cpu().tolist())
    torch.set_grad_enabled(True)

    avg_loss = total_loss / len(loader.dataset)
    macro_f1, per_class = compute_macro_f1(all_preds, all_targets, num_classes)
    accuracy = sum(p == t for p, t in zip(all_preds, all_targets)) / len(all_targets)

    return avg_loss, accuracy, macro_f1, per_class


def compute_macro_f1(preds, targets, num_classes):
    """
    Per-class precision/recall/F1 + macro-F1, computed without sklearn
    (keeps this script dependency-light — sklearn is usually present on
    Kaggle, but this avoids relying on it just for this).
    Returns (macro_f1, per_class_dict) where per_class_dict maps
    class_index -> {precision, recall, f1, support}.
    """
    per_class = {}
    f1_scores = []
    for c in range(num_classes):
        tp = sum(1 for p, t in zip(preds, targets) if p == c and t == c)
        fp = sum(1 for p, t in zip(preds, targets) if p == c and t != c)
        fn = sum(1 for p, t in zip(preds, targets) if p != c and t == c)
        support = sum(1 for t in targets if t == c)

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

        per_class[c] = {"precision": precision, "recall": recall, "f1": f1, "support": support}
        f1_scores.append(f1)

    macro_f1 = sum(f1_scores) / len(f1_scores) if f1_scores else 0.0
    return macro_f1, per_class


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    parser = argparse.ArgumentParser(description="Fine-tune EfficientNet-B0/MobileNet on the 10-class fundus dataset.")
    parser.add_argument("--split_dir", default=DEFAULT_SPLIT_DIR,
                         help=f"Path to the {{train,val,test}} split (default: {DEFAULT_SPLIT_DIR})")
    parser.add_argument("--output_dir", default=DEFAULT_OUTPUT_DIR)
    parser.add_argument("--arch", default="efficientnet_b0",
                         choices=["efficientnet_b0", "mobilenet_v2"],
                         help="mobilenet_v2 is torchvision's closest match to the project's "
                              "existing mobilenet_best.pth naming; adjust if your original "
                              "MobileNet was a different variant.")
    parser.add_argument("--epochs", type=int, default=15)
    parser.add_argument("--batch_size", type=int, default=32)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--unfreeze_last_block", action="store_true",
                         help="Opt-in: also fine-tune the last conv block, not just the "
                              "classifier head. Run with and without this flag and compare "
                              "val macro-F1 — that comparison is the point, not just picking one.")
    parser.add_argument("--run_name", default=None,
                         help="Name for this run's checkpoint/log files. Defaults to --arch "
                              "(+ '_unfrozen' if --unfreeze_last_block).")
    parser.add_argument("--num_workers", type=int, default=2)
    parser.add_argument("--seed", type=int, default=42)
    args, _unknown = parser.parse_known_args()

    torch.manual_seed(args.seed)

    run_name = args.run_name or (args.arch + ("_unfrozen" if args.unfreeze_last_block else ""))
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    if device.type == "cpu":
        print("[WARN] No GPU detected. If you enabled a Kaggle GPU accelerator, "
              "check Settings -> Accelerator in the notebook sidebar — training "
              "on CPU will be substantially slower.")

    split_dir = Path(args.split_dir)
    train_dir, val_dir = split_dir / "train", split_dir / "val"
    if not train_dir.exists() or not val_dir.exists():
        print(f"[ERROR] Expected {train_dir} and {val_dir} to exist. "
              f"Run analyze_dataset.py first to create the split.")
        sys.exit(1)

    # --- Data ---
    # Load train with transform=None here — per-class augmentation intensity
    # is applied by PerClassAugmentDataset below, not by ImageFolder directly.
    train_dataset_raw, train_empty = safe_image_folder(str(train_dir), transform=None)
    val_dataset, val_empty = safe_image_folder(str(val_dir), build_transforms(train=False))

    # ImageFolder sorts class names alphabetically to assign indices — save
    # this mapping now so eval_baseline.py and predictor.py stay consistent
    # with what this run actually trained on.
    class_to_idx = train_dataset_raw.class_to_idx
    idx_to_class = {v: k for k, v in class_to_idx.items()}
    num_classes = len(class_to_idx)
    print(f"Classes ({num_classes}): {class_to_idx}")

    if val_dataset.class_to_idx != class_to_idx:
        print("[ERROR] train/val have a DIFFERENT set of non-empty classes "
              f"(train excluded: {train_empty}, val excluded: {val_empty}). "
              "This means the model's output layer (sized from train classes) "
              "won't align with val's class indices, and val metrics for the "
              "mismatched classes would be meaningless if training proceeded. "
              "This is a direct consequence of severe class imbalance — some "
              "classes are too small to have any images in every split. "
              "Fix by either: (a) reducing val_ratio/test_ratio so more of a "
              "rare class's images stay in train, or (b) accepting those "
              "classes cannot be validated and excluding them from num_classes "
              "consistently across train/val/test — this script does not "
              "silently choose (b) for you.")
        sys.exit(1)

    heavy_class_indices = determine_heavy_classes(train_dataset_raw, threshold=100)
    print(f"Heavy augmentation applied to {len(heavy_class_indices)} minority classes: "
          f"{sorted(idx_to_class[i] for i in heavy_class_indices)}")

    train_dataset = PerClassAugmentDataset(
        train_dataset_raw,
        heavy_class_indices=heavy_class_indices,
        standard_transform=build_transforms(train=True, intensity="standard"),
        heavy_transform=build_transforms(train=True, intensity="heavy"),
    )

    sampler = build_weighted_sampler(train_dataset_raw)  # needs .samples, use the raw ImageFolder
    train_loader = DataLoader(train_dataset, batch_size=args.batch_size,
                               sampler=sampler, num_workers=args.num_workers)
    val_loader = DataLoader(val_dataset, batch_size=args.batch_size,
                             shuffle=False, num_workers=args.num_workers)

    class_weights = build_loss_class_weights(train_dataset_raw, device)
    print(f"Loss class weights (by class index): {[round(w, 3) for w in class_weights.cpu().tolist()]}")

    # --- Model ---
    model = build_model(args.arch, num_classes, args.unfreeze_last_block).to(device)
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable params: {trainable_params:,} / {total_params:,} "
          f"({100*trainable_params/total_params:.1f}%)")

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad], lr=args.lr
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=2
    )

    # --- Train loop ---
    best_val_macro_f1 = -1.0
    history = []
    ckpt_path = output_dir / f"{run_name}_best.pth"
    log_path = output_dir / f"{run_name}_training_log.json"

    print(f"\nTraining run: {run_name}")
    print(f"Checkpoint will be saved to: {ckpt_path}\n")

    for epoch in range(1, args.epochs + 1):
        t0 = time.time()
        train_loss, train_acc, train_f1, _ = run_epoch(
            model, train_loader, criterion, optimizer, device, train=True, num_classes=num_classes
        )
        val_loss, val_acc, val_f1, val_per_class = run_epoch(
            model, val_loader, criterion, optimizer, device, train=False, num_classes=num_classes
        )
        scheduler.step(val_f1)
        elapsed = time.time() - t0

        print(f"Epoch {epoch:3d}/{args.epochs}  "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.3f} train_f1={train_f1:.3f}  |  "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.3f} val_f1={val_f1:.3f}  "
              f"({elapsed:.1f}s)")

        history.append({
            "epoch": epoch, "train_loss": train_loss, "train_acc": train_acc, "train_f1": train_f1,
            "val_loss": val_loss, "val_acc": val_acc, "val_f1": val_f1,
        })

        if val_f1 > best_val_macro_f1:
            best_val_macro_f1 = val_f1
            torch.save({
                "model_state_dict": model.state_dict(),
                "arch": args.arch,
                "num_classes": num_classes,
                "class_to_idx": class_to_idx,
                "idx_to_class": idx_to_class,
                "unfreeze_last_block": args.unfreeze_last_block,
                "val_macro_f1": val_f1,
                "val_accuracy": val_acc,
                "epoch": epoch,
            }, ckpt_path)
            print(f"  -> New best (val_macro_f1={val_f1:.3f}). Saved to {ckpt_path}")

    # --- Final report: per-class breakdown WITH support counts, so it's ---
    # --- immediately visible which classes have statistically meaningful ---
    # --- numbers vs which don't (per the "report honestly" decision). ---
    print(f"\n{'='*70}")
    print(f"Best val macro-F1: {best_val_macro_f1:.3f} (epoch {history[-1]['epoch']})")
    print(f"\nFinal-epoch per-class validation breakdown:")
    print(f"{'Class':40s} {'Precision':>10s} {'Recall':>8s} {'F1':>8s} {'Support':>8s}  Note")
    for idx in sorted(val_per_class.keys()):
        cls_name = idx_to_class[idx]
        m = val_per_class[idx]
        note = ""
        if m["support"] < 10:
            note = "<- too few val samples to trust this number"
        elif m["support"] < 25:
            note = "<- low sample count, treat with caution"
        print(f"{cls_name:40s} {m['precision']:10.3f} {m['recall']:8.3f} {m['f1']:8.3f} "
              f"{m['support']:8d}  {note}")

    with open(log_path, "w") as f:
        json.dump({
            "run_name": run_name, "arch": args.arch,
            "unfreeze_last_block": args.unfreeze_last_block,
            "best_val_macro_f1": best_val_macro_f1,
            "class_to_idx": class_to_idx,
            "history": history,
            "final_val_per_class": {idx_to_class[k]: v for k, v in val_per_class.items()},
        }, f, indent=2)
    print(f"\n[OK] Saved training log -> {log_path}")
    print(f"[OK] Best checkpoint -> {ckpt_path}")
    print(f"\nNext: run this same script with --arch mobilenet_v2 for the ensemble step, "
          f"then eval_baseline.py against the held-out test split.")


if __name__ == "__main__":
    main()

Device: cuda
Classes (10): {'Central Serous Chorioretinopathy': 0, 'Diabetic Retinopathy': 1, 'Disc Edema': 2, 'Glaucoma': 3, 'Healthy': 4, 'Macular Scar': 5, 'Myopia': 6, 'Pterygium': 7, 'Retinal Detachment': 8, 'Retinitis Pigmentosa': 9}
Heavy augmentation applied to 5 minority classes: ['Central Serous Chorioretinopathy', 'Disc Edema', 'Pterygium', 'Retinal Detachment', 'Retinitis Pigmentosa']
Loss class weights (by class index): [1.262, 0.325, 1.125, 0.344, 0.394, 0.6, 0.564, 3.183, 1.132, 1.072]
Trainable params: 424,970 / 2,236,682 (19.0%)

Training run: mobilenet_unfrozen
Checkpoint will be saved to: /kaggle/working/mobilenet_unfrozen_best.pth

Epoch   1/10  train_loss=0.7022 train_acc=0.612 train_f1=0.592  |  val_loss=1.2906 val_acc=0.531 val_f1=0.540  (120.1s)
  -> New best (val_macro_f1=0.540). Saved to /kaggle/working/mobilenet_unfrozen_best.pth
Epoch   2/10  train_loss=0.4931 train_acc=0.699 train_f1=0.689  |  val_loss=1.2021 val_acc=0.574 val_f1=0.575  (123.3s)
  -> New be

In [11]:
import sys
sys.argv = [
    "train.py",
    "--arch", "efficientnet_b0",
    "--epochs", "10",
    "--unfreeze_last_block",
    "--run_name", "efficientnet_unfrozen_augmented",
]
main()

Device: cuda
Classes (10): {'Central Serous Chorioretinopathy': 0, 'Diabetic Retinopathy': 1, 'Disc Edema': 2, 'Glaucoma': 3, 'Healthy': 4, 'Macular Scar': 5, 'Myopia': 6, 'Pterygium': 7, 'Retinal Detachment': 8, 'Retinitis Pigmentosa': 9}
Heavy augmentation applied to 5 minority classes: ['Central Serous Chorioretinopathy', 'Disc Edema', 'Pterygium', 'Retinal Detachment', 'Retinitis Pigmentosa']
Loss class weights (by class index): [1.262, 0.325, 1.125, 0.344, 0.394, 0.6, 0.564, 3.183, 1.132, 1.072]
Trainable params: 424,970 / 4,020,358 (10.6%)

Training run: efficientnet_unfrozen_augmented
Checkpoint will be saved to: /kaggle/working/efficientnet_unfrozen_augmented_best.pth

Epoch   1/10  train_loss=0.8429 train_acc=0.555 train_f1=0.528  |  val_loss=1.3010 val_acc=0.514 val_f1=0.524  (120.8s)
  -> New best (val_macro_f1=0.524). Saved to /kaggle/working/efficientnet_unfrozen_augmented_best.pth
Epoch   2/10  train_loss=0.4918 train_acc=0.715 train_f1=0.698  |  val_loss=1.1854 val_acc=0

In [12]:
"""
eval_baseline.py  (Kaggle Notebook version)

Final evaluation: run trained checkpoints (from train.py) against the
REAL HELD-OUT TEST SET — not val, which has been used for every training
decision so far (checkpoint selection, frozen-vs-unfrozen comparison,
augmentation tuning). Test-set numbers are what actually belong in a CV
or interview, since val has effectively been used for model selection
throughout this project.

WHAT THIS PRODUCES:
  - Per-model (EfficientNet, MobileNet) test-set metrics: accuracy,
    macro-F1, per-class precision/recall/F1 with support counts and the
    same "too few samples to trust this" flags used during training.
  - A full confusion matrix per model, saved as both a plot and raw
    counts — this is what actually explains WHY Glaucoma has been the
    persistent weak point across every training run, rather than just
    reporting that it's weak.
  - Results saved to JSON in a format the upcoming ensemble script reuses
    directly, so ensemble.py doesn't need to reimplement per-class
    metric computation from scratch.

KAGGLE NOTES:
  - Reads checkpoints from /kaggle/working/<run_name>_best.pth (wherever
    train.py saved them in this session).
  - Reads the test split from /kaggle/working/dataset_split/test/.
  - Saves confusion matrix plots and a results JSON to /kaggle/working/.

USAGE (same pattern as train.py and analyze_dataset.py — paste into a
cell, then call main() with sys.argv in the next cell):

    import sys
    sys.argv = [
        "eval_baseline.py",
        "--checkpoints", "/kaggle/working/efficientnet_unfrozen_best.pth",
                          "/kaggle/working/mobilenet_unfrozen_best.pth",
    ]
    main()

If you ran augv2 or any other checkpoint, add its path to --checkpoints
too — the script evaluates every checkpoint you pass, independently.
"""

import argparse
import json
import sys
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

DEFAULT_SPLIT_DIR = "/kaggle/working/dataset_split"
DEFAULT_OUTPUT_DIR = "/kaggle/working"

IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".ppm", ".pgm", ".webp"}
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


# ---------------------------------------------------------------------------
# Data (mirrors train.py's safe_image_folder — kept self-contained here so
# this script doesn't depend on train.py being pasted into the same
# notebook session, since eval may run in a fresh session against saved
# checkpoints from an earlier one).
# ---------------------------------------------------------------------------

def safe_image_folder(root: str, transform):
    root_path = Path(root)
    empty_classes = []
    for d in sorted(root_path.iterdir()):
        if d.is_dir():
            has_images = any(
                f.is_file() and f.suffix.lower() in IMG_EXTENSIONS
                for f in d.rglob("*")
            )
            if not has_images:
                empty_classes.append(d.name)

    if empty_classes:
        print(f"[WARN] Empty class folder(s) in {root}: {empty_classes}. "
              f"These will be excluded from evaluation for this split.")

    if not empty_classes:
        dataset = datasets.ImageFolder(
            root, transform=transform,
            is_valid_file=lambda p: Path(p).suffix.lower() in IMG_EXTENSIONS,
        )
    else:
        dataset = _image_folder_excluding(root, transform, empty_classes)

    return dataset, empty_classes


def _image_folder_excluding(root: str, transform, exclude_classes):
    root_path = Path(root)
    tmp_classes = [d.name for d in sorted(root_path.iterdir())
                    if d.is_dir() and d.name not in exclude_classes]
    if not tmp_classes:
        raise RuntimeError(f"No non-empty class folders found under {root} "
                            f"after excluding {exclude_classes}.")
    class_to_idx = {cls: i for i, cls in enumerate(tmp_classes)}
    samples = []
    for cls in tmp_classes:
        for f in sorted((root_path / cls).rglob("*")):
            if f.is_file() and f.suffix.lower() in IMG_EXTENSIONS:
                samples.append((str(f), class_to_idx[cls]))

    ds = datasets.ImageFolder.__new__(datasets.ImageFolder)
    ds.root = str(root_path)
    ds.transform = transform
    ds.target_transform = None
    ds.loader = datasets.folder.default_loader
    ds.extensions = tuple(IMG_EXTENSIONS)
    ds.classes = tmp_classes
    ds.class_to_idx = class_to_idx
    ds.samples = samples
    ds.targets = [s[1] for s in samples]
    ds.imgs = ds.samples
    return ds


def build_eval_transform():
    """Deterministic — no augmentation. Same as train.py's val/test transform."""
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


# ---------------------------------------------------------------------------
# Model loading
# ---------------------------------------------------------------------------

def load_checkpoint_model(ckpt_path: str, device):
    """
    Loads a checkpoint saved by train.py and reconstructs the exact model
    architecture it describes (arch, num_classes, unfreeze_last_block all
    come FROM the checkpoint, not guessed — so this works regardless of
    which config produced it).
    """
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

    arch = ckpt["arch"]
    num_classes = ckpt["num_classes"]

    if arch == "efficientnet_b0":
        model = models.efficientnet_b0(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif arch == "mobilenet_v2":
        model = models.mobilenet_v2(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    else:
        raise ValueError(f"Unknown arch '{arch}' in checkpoint {ckpt_path}")

    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device)
    model.eval()

    return model, ckpt


# ---------------------------------------------------------------------------
# Evaluation
# ---------------------------------------------------------------------------

def evaluate_model(model, loader, device, num_classes):
    """
    Runs inference over the full loader and returns:
      - all_probs: [N, num_classes] softmax probabilities (needed later
        for ensembling — this is why we return probs, not just argmax)
      - all_preds, all_targets: for metrics
      - confusion matrix as a [num_classes, num_classes] tensor
        (rows = true class, cols = predicted class)
    """
    all_probs, all_preds, all_targets = [], [], []
    confusion = torch.zeros(num_classes, num_classes, dtype=torch.long)

    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            preds = probs.argmax(1)

            all_probs.append(probs.cpu())
            all_preds.extend(preds.cpu().tolist())
            all_targets.extend(targets.tolist())

            for t, p in zip(targets.tolist(), preds.cpu().tolist()):
                confusion[t, p] += 1

    all_probs = torch.cat(all_probs, dim=0)
    return all_probs, all_preds, all_targets, confusion


def compute_per_class_metrics(preds, targets, num_classes):
    """Same logic as train.py's compute_macro_f1 — kept identical for
    consistency between validation-time and test-time numbers."""
    per_class = {}
    f1_scores = []
    for c in range(num_classes):
        tp = sum(1 for p, t in zip(preds, targets) if p == c and t == c)
        fp = sum(1 for p, t in zip(preds, targets) if p == c and t != c)
        fn = sum(1 for p, t in zip(preds, targets) if p != c and t == c)
        support = sum(1 for t in targets if t == c)

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

        per_class[c] = {"precision": precision, "recall": recall, "f1": f1, "support": support}
        f1_scores.append(f1)

    macro_f1 = sum(f1_scores) / len(f1_scores) if f1_scores else 0.0
    accuracy = sum(p == t for p, t in zip(preds, targets)) / len(targets) if targets else 0.0
    return macro_f1, accuracy, per_class


def print_report(run_name, macro_f1, accuracy, per_class, idx_to_class):
    print(f"\n{'='*70}")
    print(f"{run_name} — TEST SET RESULTS")
    print(f"{'='*70}")
    print(f"Accuracy:   {accuracy:.3f}")
    print(f"Macro-F1:   {macro_f1:.3f}")
    print(f"\n{'Class':40s} {'Precision':>10s} {'Recall':>8s} {'F1':>8s} {'Support':>8s}  Note")
    for idx in sorted(per_class.keys()):
        cls_name = idx_to_class[idx]
        m = per_class[idx]
        note = ""
        if m["support"] < 10:
            note = "<- too few test samples to trust this number"
        elif m["support"] < 25:
            note = "<- low sample count, treat with caution"
        print(f"{cls_name:40s} {m['precision']:10.3f} {m['recall']:8.3f} {m['f1']:8.3f} "
              f"{m['support']:8d}  {note}")


def plot_confusion_matrix(confusion, class_names, output_path, title):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        import numpy as np
    except ImportError:
        print("[WARN] matplotlib not installed — skipping confusion matrix plot.")
        return

    cm = confusion.numpy()
    # Row-normalize so the plot shows, per true class, what fraction of
    # predictions went where — raw counts alone are hard to read across
    # classes with very different support sizes.
    row_sums = cm.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1  # avoid divide-by-zero for any 0-support class
    cm_normalized = cm / row_sums

    fig, ax = plt.subplots(figsize=(10, 9))
    im = ax.imshow(cm_normalized, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title(f"{title}\n(row-normalized: fraction of each true class's predictions)")

    for i in range(len(class_names)):
        for j in range(len(class_names)):
            count = cm[i, j]
            if count > 0:
                text_color = "white" if cm_normalized[i, j] > 0.5 else "black"
                ax.text(j, i, str(int(count)), ha="center", va="center",
                        color=text_color, fontsize=8)

    fig.colorbar(im, ax=ax, label="Fraction of true class")
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()
    print(f"[OK] Saved confusion matrix -> {output_path}")


def summarize_top_confusions(confusion, idx_to_class, top_n=5):
    """
    Prints the top off-diagonal confusion pairs — i.e. which classes are
    most often mistaken for which other classes. This is what actually
    explains a weak class's low recall, rather than just reporting the
    number.
    """
    cm = confusion.clone()
    num_classes = cm.shape[0]
    pairs = []
    for i in range(num_classes):
        for j in range(num_classes):
            if i != j and cm[i, j] > 0:
                pairs.append((cm[i, j].item(), idx_to_class[i], idx_to_class[j]))
    pairs.sort(reverse=True)

    print(f"\nTop confusions (true class -> predicted class, count):")
    for count, true_cls, pred_cls in pairs[:top_n]:
        print(f"  {true_cls:35s} -> {pred_cls:35s}  ({count} test images)")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    parser = argparse.ArgumentParser(description="Evaluate trained checkpoints on the held-out test set.")
    parser.add_argument("--checkpoints", nargs="+", required=True,
                         help="One or more checkpoint paths (from train.py) to evaluate independently. "
                              "e.g. --checkpoints /kaggle/working/efficientnet_unfrozen_best.pth "
                              "/kaggle/working/mobilenet_unfrozen_best.pth")
    parser.add_argument("--split_dir", default=DEFAULT_SPLIT_DIR)
    parser.add_argument("--output_dir", default=DEFAULT_OUTPUT_DIR)
    parser.add_argument("--batch_size", type=int, default=32)
    parser.add_argument("--num_workers", type=int, default=2)
    args, _unknown = parser.parse_known_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    test_dir = Path(args.split_dir) / "test"
    if not test_dir.exists():
        print(f"[ERROR] Test split not found at {test_dir}. Run analyze_dataset.py first.")
        sys.exit(1)

    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    all_results = {}

    for ckpt_path in args.checkpoints:
        ckpt_path = Path(ckpt_path)
        if not ckpt_path.exists():
            print(f"[ERROR] Checkpoint not found: {ckpt_path}. Skipping.")
            continue

        run_name = ckpt_path.stem.replace("_best", "")
        print(f"\nLoading checkpoint: {ckpt_path}")
        model, ckpt = load_checkpoint_model(str(ckpt_path), device)

        class_to_idx = ckpt["class_to_idx"]
        idx_to_class = {int(v): k for k, v in ckpt["idx_to_class"].items()} \
            if isinstance(list(ckpt["idx_to_class"].keys())[0], str) else ckpt["idx_to_class"]
        num_classes = ckpt["num_classes"]

        # Build test dataset — use the SAME class_to_idx as training so
        # label indices line up with what the model was actually trained on,
        # rather than trusting ImageFolder to re-derive identical indices
        # (it would, since both use alphabetical sort, but this is safer
        # and self-documenting).
        test_dataset, test_empty = safe_image_folder(str(test_dir), build_eval_transform())
        if test_dataset.class_to_idx != class_to_idx:
            print(f"[WARN] Test set class_to_idx differs from checkpoint's training "
                  f"class_to_idx (test excluded: {test_empty}). Results for missing "
                  f"classes will show 0 support — check before trusting this run's numbers.")

        test_loader = DataLoader(test_dataset, batch_size=args.batch_size,
                                  shuffle=False, num_workers=args.num_workers)

        probs, preds, targets, confusion = evaluate_model(model, test_loader, device, num_classes)
        macro_f1, accuracy, per_class = compute_per_class_metrics(preds, targets, num_classes)

        print_report(run_name, macro_f1, accuracy, per_class, idx_to_class)
        summarize_top_confusions(confusion, idx_to_class)

        cm_path = output_dir / f"{run_name}_test_confusion_matrix.png"
        plot_confusion_matrix(confusion, [idx_to_class[i] for i in range(num_classes)],
                               cm_path, f"{run_name} — Test Set Confusion Matrix")

        all_results[run_name] = {
            "checkpoint": str(ckpt_path),
            "arch": ckpt["arch"],
            "unfreeze_last_block": ckpt.get("unfreeze_last_block"),
            "val_macro_f1_at_training_time": ckpt.get("val_macro_f1"),
            "test_accuracy": accuracy,
            "test_macro_f1": macro_f1,
            "test_per_class": {idx_to_class[k]: v for k, v in per_class.items()},
            "class_to_idx": class_to_idx,
        }

        # Save raw probabilities + targets for this model — the ensemble
        # script needs these to average softmax outputs across models
        # without re-running inference from scratch.
        probs_path = output_dir / f"{run_name}_test_probs.pt"
        torch.save({
            "probs": probs, "targets": torch.tensor(targets),
            "class_to_idx": class_to_idx, "idx_to_class": idx_to_class,
        }, probs_path)
        print(f"[OK] Saved test-set probabilities -> {probs_path} (for ensemble.py)")

    # --- Cross-model comparison summary ---
    if len(all_results) > 1:
        print(f"\n{'='*70}")
        print("CROSS-MODEL COMPARISON (test set)")
        print(f"{'='*70}")
        print(f"{'Model':30s} {'Accuracy':>10s} {'Macro-F1':>10s}")
        for run_name, r in all_results.items():
            print(f"{run_name:30s} {r['test_accuracy']:10.3f} {r['test_macro_f1']:10.3f}")

    results_path = output_dir / "eval_baseline_results.json"
    with open(results_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\n[OK] Saved full results -> {results_path}")
    print("\nNext: ensemble.py will load the *_test_probs.pt files saved here "
          "to average softmax outputs across models without re-running inference.")


if __name__ == "__main__":
    main()

usage: train.py [-h] --checkpoints CHECKPOINTS [CHECKPOINTS ...]
                [--split_dir SPLIT_DIR] [--output_dir OUTPUT_DIR]
                [--batch_size BATCH_SIZE] [--num_workers NUM_WORKERS]
train.py: error: the following arguments are required: --checkpoints
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/lib/python3.12/argparse.py", line 1943, in _parse_known_args2
    namespace, args = self._parse_known_args(args, namespace, intermixed)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/argparse.py", line 2230, in _parse_known_args
    raise ArgumentError(None, _('the following arguments are required: %s') %
argparse.ArgumentError: the following arguments are required: --checkpoints

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_58/2336823473.py", line 412, in <cell line: 0>
    main()
  File "/tmp/ipykernel_58/2336823473.py", line 319, in main
    args, _unknown = parser.parse_known_args()
                     ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/l

TypeError: object of type 'NoneType' has no len()

In [13]:
import sys
sys.argv = [
    "eval_baseline.py",
    "--checkpoints",
    "/kaggle/working/efficientnet_unfrozen_best.pth",
    "/kaggle/working/mobilenet_unfrozen_best.pth",
]
main()

Device: cuda

Loading checkpoint: /kaggle/working/efficientnet_unfrozen_best.pth

efficientnet_unfrozen — TEST SET RESULTS
Accuracy:   0.635
Macro-F1:   0.625

Class                                     Precision   Recall       F1  Support  Note
Central Serous Chorioretinopathy              0.224    0.688    0.338       16  <- low sample count, treat with caution
Diabetic Retinopathy                          0.931    0.775    0.846      227  
Disc Edema                                    0.395    0.750    0.517       20  <- low sample count, treat with caution
Glaucoma                                      0.752    0.389    0.513      203  
Healthy                                       0.631    0.652    0.641      155  
Macular Scar                                  0.414    0.426    0.420       68  
Myopia                                        0.484    0.827    0.611       75  
Pterygium                                     1.000    1.000    1.000        4  <- too few test samples to tru

In [14]:
"""
ensemble.py  (Kaggle Notebook version)

Combines EfficientNet + MobileNet predictions by averaging their softmax
outputs on the test set — loaded from the *_test_probs.pt files
eval_baseline.py already saved, so this does NOT re-run inference.

IMPORTANT CONTEXT FROM eval_baseline.py's RESULTS (read before interpreting
this script's output): the confusion matrix showed BOTH models confuse
Glaucoma with Healthy and Myopia in the same direction — not different,
complementary mistakes. Ensembling helps most when models fail differently;
here the dominant failure mode is shared. So: don't expect this to fix
Glaucoma specifically. Real gains, if any, are more likely to show up as
averaged-out noise on other classes, or a modest overall bump. This script
reports honestly whether that happened or not — it does not assume the
ensemble will win.

WHAT THIS PRODUCES:
  - Simple average ensemble: (prob_effnet + prob_mobilenet) / 2
  - Weighted average ensemble: weighted toward whichever solo model had
    the higher test macro-F1 (from eval_baseline_results.json), since an
    equal-weight average isn't obviously right when the two models aren't
    equally good.
  - Same per-class report format as eval_baseline.py, plus a final
    3-way comparison table (EfficientNet solo vs MobileNet solo vs
    ensemble) so the result is directly comparable to what you already have.

KAGGLE NOTES:
  - Reads /kaggle/working/<run_name>_test_probs.pt (from eval_baseline.py)
  - Reads /kaggle/working/eval_baseline_results.json for each model's solo
    test macro-F1 (used to set weighted-ensemble weights)
  - Saves a confusion matrix + results JSON to /kaggle/working/

USAGE:
    import sys
    sys.argv = [
        "ensemble.py",
        "--probs", "/kaggle/working/efficientnet_unfrozen_test_probs.pt",
                   "/kaggle/working/mobilenet_unfrozen_test_probs.pt",
    ]
    main()

Exactly two --probs paths are expected (this script does pairwise
ensembling; extend the average step yourself if you train a third model).
"""

import argparse
import json
import sys
from pathlib import Path

import torch

DEFAULT_OUTPUT_DIR = "/kaggle/working"


# ---------------------------------------------------------------------------
# Metrics (identical logic to train.py / eval_baseline.py, kept consistent
# on purpose so numbers are directly comparable across all three scripts)
# ---------------------------------------------------------------------------

def compute_per_class_metrics(preds, targets, num_classes):
    per_class = {}
    f1_scores = []
    for c in range(num_classes):
        tp = sum(1 for p, t in zip(preds, targets) if p == c and t == c)
        fp = sum(1 for p, t in zip(preds, targets) if p == c and t != c)
        fn = sum(1 for p, t in zip(preds, targets) if p != c and t == c)
        support = sum(1 for t in targets if t == c)

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

        per_class[c] = {"precision": precision, "recall": recall, "f1": f1, "support": support}
        f1_scores.append(f1)

    macro_f1 = sum(f1_scores) / len(f1_scores) if f1_scores else 0.0
    accuracy = sum(p == t for p, t in zip(preds, targets)) / len(targets) if targets else 0.0
    return macro_f1, accuracy, per_class


def compute_confusion(preds, targets, num_classes):
    confusion = torch.zeros(num_classes, num_classes, dtype=torch.long)
    for t, p in zip(targets, preds):
        confusion[t, p] += 1
    return confusion


def print_report(run_name, macro_f1, accuracy, per_class, idx_to_class):
    print(f"\n{'='*70}")
    print(f"{run_name} — TEST SET RESULTS")
    print(f"{'='*70}")
    print(f"Accuracy:   {accuracy:.3f}")
    print(f"Macro-F1:   {macro_f1:.3f}")
    print(f"\n{'Class':40s} {'Precision':>10s} {'Recall':>8s} {'F1':>8s} {'Support':>8s}  Note")
    for idx in sorted(per_class.keys()):
        cls_name = idx_to_class[idx]
        m = per_class[idx]
        note = ""
        if m["support"] < 10:
            note = "<- too few test samples to trust this number"
        elif m["support"] < 25:
            note = "<- low sample count, treat with caution"
        print(f"{cls_name:40s} {m['precision']:10.3f} {m['recall']:8.3f} {m['f1']:8.3f} "
              f"{m['support']:8d}  {note}")


def summarize_top_confusions(confusion, idx_to_class, top_n=5):
    cm = confusion.clone()
    num_classes = cm.shape[0]
    pairs = []
    for i in range(num_classes):
        for j in range(num_classes):
            if i != j and cm[i, j] > 0:
                pairs.append((cm[i, j].item(), idx_to_class[i], idx_to_class[j]))
    pairs.sort(reverse=True)

    print(f"\nTop confusions (true class -> predicted class, count):")
    for count, true_cls, pred_cls in pairs[:top_n]:
        print(f"  {true_cls:35s} -> {pred_cls:35s}  ({count} test images)")


def plot_confusion_matrix(confusion, class_names, output_path, title):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        print("[WARN] matplotlib not installed — skipping confusion matrix plot.")
        return

    cm = confusion.numpy()
    row_sums = cm.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    cm_normalized = cm / row_sums

    fig, ax = plt.subplots(figsize=(10, 9))
    im = ax.imshow(cm_normalized, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title(f"{title}\n(row-normalized: fraction of each true class's predictions)")

    for i in range(len(class_names)):
        for j in range(len(class_names)):
            count = cm[i, j]
            if count > 0:
                text_color = "white" if cm_normalized[i, j] > 0.5 else "black"
                ax.text(j, i, str(int(count)), ha="center", va="center",
                        color=text_color, fontsize=8)

    fig.colorbar(im, ax=ax, label="Fraction of true class")
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()
    print(f"[OK] Saved confusion matrix -> {output_path}")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    parser = argparse.ArgumentParser(description="Ensemble two models by averaging saved test-set softmax probabilities.")
    parser.add_argument("--probs", nargs=2, required=True, metavar=("MODEL_A_PROBS", "MODEL_B_PROBS"),
                         help="Exactly two *_test_probs.pt paths saved by eval_baseline.py")
    parser.add_argument("--results_json", default=None,
                         help="Path to eval_baseline_results.json, used to set weighted-ensemble "
                              f"weights from each model's solo test macro-F1. Defaults to "
                              f"{DEFAULT_OUTPUT_DIR}/eval_baseline_results.json if present; if not "
                              f"found, weighted ensemble is skipped (simple average only).")
    parser.add_argument("--output_dir", default=DEFAULT_OUTPUT_DIR)
    args, _unknown = parser.parse_known_args()

    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    path_a, path_b = args.probs
    print(f"Loading: {path_a}")
    data_a = torch.load(path_a, map_location="cpu", weights_only=False)
    print(f"Loading: {path_b}")
    data_b = torch.load(path_b, map_location="cpu", weights_only=False)

    probs_a, targets_a = data_a["probs"], data_a["targets"]
    probs_b, targets_b = data_b["probs"], data_b["targets"]
    class_to_idx = data_a["class_to_idx"]
    idx_to_class = data_a["idx_to_class"]
    num_classes = len(class_to_idx)

    # --- Sanity checks before combining anything ---
    if data_a["class_to_idx"] != data_b["class_to_idx"]:
        print("[ERROR] The two models have DIFFERENT class_to_idx mappings. "
              "Averaging their softmax outputs directly would silently combine "
              "the wrong class indices with each other. This would happen if "
              "the two models were trained on different splits/class sets. "
              "Cannot proceed safely — stopping.")
        sys.exit(1)

    if not torch.equal(targets_a, targets_b):
        print("[ERROR] The two models' saved targets differ, meaning they were "
              "evaluated on different test-set orderings or subsets. Averaging "
              "their probabilities would align the wrong predictions to the "
              "wrong ground truth. This would happen if eval_baseline.py was "
              "run with different --num_workers/shuffle settings between runs, "
              "or against different test splits. Cannot proceed safely — stopping.")
        sys.exit(1)

    targets = targets_a.tolist()
    print(f"\nLoaded {len(targets)} test samples, {num_classes} classes — targets match, proceeding.")

    all_results = {}

    # --- Simple average ensemble ---
    probs_simple = (probs_a + probs_b) / 2.0
    preds_simple = probs_simple.argmax(1).tolist()
    macro_f1_simple, acc_simple, per_class_simple = compute_per_class_metrics(preds_simple, targets, num_classes)
    print_report("ensemble_simple_average", macro_f1_simple, acc_simple, per_class_simple, idx_to_class)
    confusion_simple = compute_confusion(preds_simple, targets, num_classes)
    summarize_top_confusions(confusion_simple, idx_to_class)
    plot_confusion_matrix(confusion_simple, [idx_to_class[i] for i in range(num_classes)],
                           output_dir / "ensemble_simple_test_confusion_matrix.png",
                           "Ensemble (simple average) — Test Set Confusion Matrix")
    all_results["ensemble_simple_average"] = {
        "test_accuracy": acc_simple, "test_macro_f1": macro_f1_simple,
        "test_per_class": {idx_to_class[k]: v for k, v in per_class_simple.items()},
    }

    # --- Weighted average ensemble (weight by each solo model's test macro-F1) ---
    results_json_path = Path(args.results_json) if args.results_json else output_dir / "eval_baseline_results.json"
    if results_json_path.exists():
        with open(results_json_path) as f:
            solo_results = json.load(f)

        # Match solo results to probs files by run_name embedded in the filename
        run_name_a = Path(path_a).stem.replace("_test_probs", "")
        run_name_b = Path(path_b).stem.replace("_test_probs", "")

        f1_a = solo_results.get(run_name_a, {}).get("test_macro_f1")
        f1_b = solo_results.get(run_name_b, {}).get("test_macro_f1")

        if f1_a is not None and f1_b is not None and (f1_a + f1_b) > 0:
            weight_a = f1_a / (f1_a + f1_b)
            weight_b = f1_b / (f1_a + f1_b)
            print(f"\nWeighted ensemble weights (from solo test macro-F1): "
                  f"{run_name_a}={weight_a:.3f}, {run_name_b}={weight_b:.3f}")

            probs_weighted = probs_a * weight_a + probs_b * weight_b
            preds_weighted = probs_weighted.argmax(1).tolist()
            macro_f1_weighted, acc_weighted, per_class_weighted = compute_per_class_metrics(
                preds_weighted, targets, num_classes
            )
            print_report("ensemble_weighted_average", macro_f1_weighted, acc_weighted,
                         per_class_weighted, idx_to_class)
            confusion_weighted = compute_confusion(preds_weighted, targets, num_classes)
            summarize_top_confusions(confusion_weighted, idx_to_class)
            plot_confusion_matrix(confusion_weighted, [idx_to_class[i] for i in range(num_classes)],
                                   output_dir / "ensemble_weighted_test_confusion_matrix.png",
                                   "Ensemble (weighted average) — Test Set Confusion Matrix")
            all_results["ensemble_weighted_average"] = {
                "weights": {run_name_a: weight_a, run_name_b: weight_b},
                "test_accuracy": acc_weighted, "test_macro_f1": macro_f1_weighted,
                "test_per_class": {idx_to_class[k]: v for k, v in per_class_weighted.items()},
            }
        else:
            print(f"\n[WARN] Could not find test_macro_f1 for both '{run_name_a}' and '{run_name_b}' "
                  f"in {results_json_path} — skipping weighted ensemble. Check that the run_name "
                  f"embedded in your probs filenames matches the keys in eval_baseline_results.json.")

        # --- Final comparison across everything we have ---
        print(f"\n{'='*70}")
        print("FULL COMPARISON (test set)")
        print(f"{'='*70}")
        print(f"{'Model':30s} {'Accuracy':>10s} {'Macro-F1':>10s}")
        for run_name in (run_name_a, run_name_b):
            r = solo_results.get(run_name, {})
            if "test_accuracy" in r:
                print(f"{run_name:30s} {r['test_accuracy']:10.3f} {r['test_macro_f1']:10.3f}")
        print(f"{'ensemble_simple_average':30s} {acc_simple:10.3f} {macro_f1_simple:10.3f}")
        if "ensemble_weighted_average" in all_results:
            w = all_results["ensemble_weighted_average"]
            print(f"{'ensemble_weighted_average':30s} {w['test_accuracy']:10.3f} {w['test_macro_f1']:10.3f}")

        best_key = max(
            [(run_name_a, solo_results.get(run_name_a, {}).get("test_macro_f1", -1)),
             (run_name_b, solo_results.get(run_name_b, {}).get("test_macro_f1", -1)),
             ("ensemble_simple_average", macro_f1_simple)]
            + ([("ensemble_weighted_average", all_results["ensemble_weighted_average"]["test_macro_f1"])]
               if "ensemble_weighted_average" in all_results else []),
            key=lambda x: x[1],
        )
        print(f"\nBest by macro-F1: {best_key[0]} ({best_key[1]:.3f})")
        if best_key[0] in (run_name_a, run_name_b):
            print("Note: the ensemble did NOT beat the better solo model on this run. "
                  "Given both models shared the same dominant failure mode (Glaucoma "
                  "confused with Healthy/Myopia, per eval_baseline.py's confusion "
                  "matrix), this is a plausible honest outcome, not a bug — report it "
                  "as such rather than only reporting whichever ensemble number looks best.")
    else:
        print(f"\n[WARN] {results_json_path} not found — skipping weighted ensemble "
              f"and full comparison table. Simple-average results above are still valid.")

    results_path = output_dir / "ensemble_results.json"
    with open(results_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\n[OK] Saved ensemble results -> {results_path}")


if __name__ == "__main__":
    main()

usage: eval_baseline.py [-h] --probs MODEL_A_PROBS MODEL_B_PROBS
                        [--results_json RESULTS_JSON]
                        [--output_dir OUTPUT_DIR]
eval_baseline.py: error: the following arguments are required: --probs
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/lib/python3.12/argparse.py", line 1943, in _parse_known_args2
    namespace, args = self._parse_known_args(args, namespace, intermixed)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/argparse.py", line 2230, in _parse_known_args
    raise ArgumentError(None, _('the following arguments are required: %s') %
argparse.ArgumentError: the following arguments are required: --probs

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_58/3471506785.py", line 313, in <cell line: 0>
    main()
  File "/tmp/ipykernel_58/3471506785.py", line 177, in main
    args, _unknown = parser.parse_known_args()
                     ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/pyt

TypeError: object of type 'NoneType' has no len()

In [15]:
import sys
sys.argv = [
    "ensemble.py",
    "--probs",
    "/kaggle/working/efficientnet_unfrozen_test_probs.pt",
    "/kaggle/working/mobilenet_unfrozen_test_probs.pt",
]
main()

Loading: /kaggle/working/efficientnet_unfrozen_test_probs.pt
Loading: /kaggle/working/mobilenet_unfrozen_test_probs.pt

Loaded 810 test samples, 10 classes — targets match, proceeding.

ensemble_simple_average — TEST SET RESULTS
Accuracy:   0.710
Macro-F1:   0.705

Class                                     Precision   Recall       F1  Support  Note
Central Serous Chorioretinopathy              0.393    0.688    0.500       16  <- low sample count, treat with caution
Diabetic Retinopathy                          0.929    0.868    0.897      227  
Disc Edema                                    0.516    0.800    0.627       20  <- low sample count, treat with caution
Glaucoma                                      0.850    0.448    0.587      203  
Healthy                                       0.652    0.761    0.702      155  
Macular Scar                                  0.521    0.544    0.532       68  
Myopia                                        0.516    0.880    0.650       75  
Pter